In [21]:
%%time
import numpy as np
import h5py
import re
import csv
import pandas as pd

##### IMPORT ARCHS4 H5PY DATA #####  

root = "/home/lwu/work/NarrativeGenomics/region_narration/code/pipeline/ARCHS4_LW/"
file_path = "refining/data_slash_results/"
filename = root + file_path + 'archs4_gene_human_v2.1_normalized_LW20230126.h5'

with h5py.File(filename, 'r') as f:
    gene_names = f['meta/genes/all_gene_names'][:]
arch_gene_names = [x.decode('UTF-8') for x in gene_names]

##### IMPORT MATCHED GENES DATA ##### 

file_path = "/refining/gene_length_correction/match_genes/"
filename = root + file_path + "archs4_TLF_geneLengthData.csv"
matched_genes_df = pd.read_csv(filename)

CPU times: user 96.7 ms, sys: 14.4 ms, total: 111 ms
Wall time: 126 ms


In [34]:
print("ARCHS4 Gene Names:", arch_gene_names[0:3])
print("ARCHS4 Gene Size:", len(arch_gene_names))
print()
print("ARCHS4 Matched Genes Size:",len(matched_genes_df))
print("ARCHS4 Matched Genes Size:",matched_genes_df['archs4_gene_name'].tolist()[0:3])

ARCHS4 Gene Names: ['A1BG', 'A1BG-AS1', 'A1CF']
ARCHS4 Gene Size: 57447

ARCHS4 Matched Genes Size: 57455
ARCHS4 Matched Genes Size: ['A1BG', 'A1BG-AS1', 'A1CF']


In [33]:
#**** CHECK SIZE DIFFERENCE ****# 
df_archs4_gene_names = matched_genes_df['archs4_gene_name'].tolist()

diff1 = set(df_archs4_gene_names) - set(arch_gene_names)
diff2 = set(arch_gene_names) - set(df_archs4_gene_names)

print('Items in list1 but not in list2:', diff1)
print('Items in list2 but not in list1:', diff2)
print("NO DIFFERENCE, MUST BE REPEATS IN df_archs4_gene_names")
print()

#**** CHECK df_archs4_gene_names REPEATS ****# 
if len(set(df_archs4_gene_names)) < len(df_archs4_gene_names):
    print('There are repeated elements in list1')
else:
    print('There are no repeated elements in list1')

#**** MORE INFO ON REPEATS (Gene name, #repeats, indxs) ****# 
repeated = {}
for i, x in enumerate(df_archs4_gene_names):
    if x in repeated:
        repeated[x]['count'] += 1
        repeated[x]['indices'].append(i)
    else:
        repeated[x] = {'count': 1, 'indices': [i]}

for x in repeated:
    if repeated[x]['count'] > 1:
        print(f"The element '{x}' is repeated {repeated[x]['count']} times at indices: {repeated[x]['indices']}")

Items in list1 but not in list2: set()
Items in list2 but not in list1: set()
NO DIFFERENCE, MUST BE REPEATS IN df_archs4_gene_names

There are repeated elements in list1
The element 'HERC3' is repeated 2 times at indices: [28469, 28470]
The element 'LINC03021' is repeated 2 times at indices: [33917, 33918]
The element 'LINC03023' is repeated 2 times at indices: [33920, 33921]
The element 'LINC03025' is repeated 2 times at indices: [33922, 33923]
The element 'MKKS' is repeated 3 times at indices: [37057, 37058, 37059]
The element 'NPEPPSP1' is repeated 2 times at indices: [39119, 39120]
The element 'TNFRSF10A-DT' is repeated 2 times at indices: [53911, 53912]


In [40]:
matched_genes_df.head

<bound method NDFrame.head of                gene_id archs4_gene_name tlf_gene_name  gene_length
0      ENSG00000121410             A1BG          A1BG         3374
1      ENSG00000268895         A1BG-AS1      A1BG-AS1         2409
2      ENSG00000148584             A1CF          A1CF         9208
3      ENSG00000175899              A2M           A2M         4574
4      ENSG00000245105          A2M-AS1       A2M-AS1         2928
...                ...              ...           ...          ...
57450  ENSG00000162378           ZYG11B        ZYG11B         8129
57451  ENSG00000159840              ZYX           ZYX         2218
57452  ENSG00000274572            ZYXP1         ZYXP1          117
57453  ENSG00000074755            ZZEF1         ZZEF1        11411
57454  ENSG00000036549             ZZZ3          ZZZ3         6397

[57455 rows x 4 columns]>

In [43]:
#**** GET GENE ID LIST ****# 
df_gene_id = matched_genes_df['gene_id'].tolist()
print("MATCHED GENES ID (57447):", len(set(df_gene_id)))
print(df_gene_id[0:3])

# FAILED: Size difference, revise csv file 

MATCHED GENES ID (57447): 57438
['ENSG00000121410', 'ENSG00000268895', 'ENSG00000148584']


In [45]:
#**** GET GENE ID LIST #2 ****# 

file_path = "/refining/gene_length_correction/match_genes/"
filename = root + file_path + "archs4_TLF_geneLengthData_averagedRepeats_LW20230416.csv"
matched_genes_df = pd.read_csv(filename)

df_gene_id = matched_genes_df['gene_id'].tolist()
print("MATCHED GENES ID (57447):", len(df_gene_id))
print(df_gene_id[0:3])

MATCHED GENES ID (57447): 57447
['ENSG00000121410', 'ENSG00000268895', 'ENSG00000148584']


In [ ]:
#----  INSERT GENE IDS ----# 
root = "/home/lwu/work/NarrativeGenomics/region_narration/code/pipeline/ARCHS4_LW/"
file_path = "refining/data_slash_results/"
filename = root + file_path + 'archs4_gene_human_v2.1_normalized_LW20230126.h5'

file = h5py.File(filename, "a")

# create the pathway "meta/genes/" if it does not exist
if "meta" not in file:
    file.create_group("meta")
if "genes" not in file["meta"]:
    print("Error")
    
group = f['meta/genes']
print(group.keys())

# write the list to the dataset in the pathway "meta/genes/"
file["meta/genes/gene_id"] = df_gene_id

group = f['meta/genes']
print(group.keys())

# close the h5py file
file.close()
